#### Helpers

In [ ]:
# Running covariance computation

import numpy as np


class OnlineCovariance:
    """
    A class to calculate the mean and the covariance matrix
    of the incrementally added, n-dimensional data.
    """

    def __init__(self, order):
        """
        Parameters
        ----------
        order: int, The order (=="number of features") of the incrementally added
        dataset and of the resulting covariance matrix.
        """
        self._order = order
        self._shape = (order, order)
        self._identity = np.identity(order)
        self._ones = np.ones(order)
        self._count = 0
        self._mean = np.zeros(order)
        self._cov = np.zeros(self._shape)

    @property
    def count(self):
        """
        int, The number of observations that has been added
        to this instance of OnlineCovariance.
        """
        return self._count

    @property
    def mean(self):
        """
        double, The mean of the added data.
        """
        return self._mean

    @property
    def cov(self):
        """
        array_like, The covariance matrix of the added data.
        """
        return self._cov

    @property
    def corrcoef(self):
        """
        array_like, The normalized covariance matrix of the added data.
        Consists of the Pearson Correlation Coefficients of the data's features.
        """
        if self._count < 1:
            return None
        variances = np.diagonal(self._cov)
        denomiator = np.sqrt(variances[np.newaxis, :] * variances[:, np.newaxis])
        return self._cov / denomiator

    def add(self, observation):
        """
        Add the given observation to this object.

        Parameters
        ----------
        observation: array_like, The observation to add.
        """
        if self._order != len(observation):
            raise ValueError(f"Observation to add must be of size {self._order}")

        self._count += 1
        delta_at_nMin1 = np.array(observation - self._mean)
        self._mean += delta_at_nMin1 / self._count
        weighted_delta_at_n = np.array(observation - self._mean) / self._count
        shp = (self._order, self._order)
        D_at_n = np.broadcast_to(weighted_delta_at_n, self._shape).T
        D = (delta_at_nMin1 * self._identity).dot(D_at_n.T)
        self._cov = self._cov * (self._count - 1) / self._count + D

    def merge(self, other):
        """
        Merges the current object and the given other object into a new OnlineCovariance object.

        Parameters
        ----------
        other: OnlineCovariance, The other OnlineCovariance to merge this object with.

        Returns
        -------
        OnlineCovariance
        """
        if other._order != self._order:
            raise ValueError(
                f"""
                   Cannot merge two OnlineCovariances with different orders.
                   ({self._order} != {other._order})
                   """
            )

        merged_cov = OnlineCovariance(self._order)
        merged_cov._count = self.count + other.count
        count_corr = (other.count * self.count) / merged_cov._count
        merged_cov._mean = (
            self.mean / other.count + other.mean / self.count
        ) * count_corr
        flat_mean_diff = self._mean - other._mean
        shp = (self._order, self._order)
        mean_diffs = np.broadcast_to(flat_mean_diff, self._shape).T
        merged_cov._cov = (
            self._cov * self.count
            + other._cov * other._count
            + mean_diffs * mean_diffs.T * count_corr
        ) / merged_cov.count
        return merged_cov



#### Save layer-wise covariance

In [ ]:
# Load model for Cars
import torch
import certifi
import os
import os.path as osp
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))
os.environ['SSL_CERT_FILE'] = certifi.where()
os.environ["OPENCLIP_CACHEDIR"] = os.environ["SCRATCH"] + "/openclip"

from src.datasets.registry import get_dataset
from src.datasets.common import get_dataloader_v2, maybe_dictionarize
from src.models.modeling import ImageClassifier, ImageEncoder
from src.models.heads import build_classification_head
from src.datasets.templates import get_templates

# HPs
DATASET_NAME = "Cars"
MODEL_NAME = "ViT-B-16"
DATA_LOCATION = "datasets"
BATCH_SIZE = 32
MODELS_ROOT = os.path.expanduser("models/checkpoints")

root_dir = osp.join(os.environ.get("SCRATCH", ""), "ties", "exp_out", "training", MODEL_NAME)
task_to_model_dict = {
    'qasc': osp.join(root_dir, "qasc", "best_model.pt"),
    'quartz': osp.join(root_dir, "quartz", "best_model.pt")
}
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


# Build model
template = get_templates(DATASET_NAME)
image_encoder = ImageEncoder(MODEL_NAME, keep_lang=True)
classification_head = build_classification_head(image_encoder.model, DATASET_NAME, template, DATA_LOCATION, device)
model = ImageClassifier(image_encoder, classification_head)

dataset = get_dataset(
    DATASET_NAME,
    model.val_preprocess,
    location=DATA_LOCATION,
    batch_size=BATCH_SIZE,
)
dataloader = get_dataloader_v2(dataset, is_train=False, image_encoder=None, batch_size=BATCH_SIZE, device=device)


In [ ]:
# Get finetuned params
from src.models.task_vectors import NonLinearTaskVector

def add_tv_to_model(model, tv):
    new_dict = {}
    skipped = []
    added = []
    pt_dict = model.state_dict()
    tv_dict = tv.vector
    for k in pt_dict.keys():
        if k in tv_dict:
            new_dict[k] = pt_dict[k].to(device) + tv_dict[k].to(device)
            added.append(k)
        else:
            skipped.append(k)
            new_dict[k] = pt_dict[k]
    return new_dict, added, skipped


# Calculate task vectors
pretrained_checkpoint = f"{MODELS_ROOT}/{MODEL_NAME}/MNISTVal/nonlinear_zeroshot.pt"
finetuned_checkpoint = f"{MODELS_ROOT}/{MODEL_NAME}/{DATASET_NAME}Val/nonlinear_finetuned.pt"
tv = NonLinearTaskVector(MODEL_NAME, pretrained_checkpoint, finetuned_checkpoint)

# Load tv into model
new_dict, added, skipped = add_tv_to_model(model.image_encoder, tv)
model.image_encoder.load_state_dict(new_dict)
print("Added layers:", added)
print("Skipped layers:", skipped)

In [ ]:
import torch
import itertools

# Compute running covariance of activations
stats = {}  # layer_name -> hook_result
handles = []  # references to hooks
MAX_HOOKS = 50
MAX_BATCHES = 20


def hook(name):
    # Hook gets (module, input, output)
    def h(mod, _inp, out):
        # print("out.shape", out.shape)
        inp = _inp[0]
        if torch.is_tensor(inp):
            print("inp.shape", inp.shape)
            B, T, D = inp.shape
            if name not in stats:
                ocov = OnlineCovariance(D)
                stats[name] = ocov
            ocov = stats[name]
            for i in range(B):
                j = torch.randint(0, T, (1,)).item()
                v = inp[i, j].cpu().detach().numpy()
                # normalize v
                # v = v / np.linalg.norm(v)
                ocov.add(v)
            stats[name] = ocov
    return h

num_registered = 0
for i, (c_mat_name, m) in enumerate(model.named_modules()):
    if isinstance(m, torch.nn.Linear) or isinstance(m, torch.nn.MultiheadAttention):
        print("Registering hook for", c_mat_name)
        h = m.register_forward_hook(hook(c_mat_name))
        handles.append(h)
        if num_registered > MAX_HOOKS:
            break
        num_registered += 1

model.to(device)
with torch.no_grad():
    for i, batch in enumerate(dataloader):
        print("Processing batch", i)
        x = maybe_dictionarize(batch)["images"]
        model(x.to(device))
        if i > MAX_BATCHES:
            break